# SamplerV2 and finite-shot counts

Compare Qiskit's StatevectorSampler with MettleQSamplerV2 using the same Bell-state sampling contract.

## What you will learn

- How to express this workflow with Qiskit's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    print_scaling_table,
    qiskit_selection,
    total_variation_distance,
)

## 1. Define the quantum problem

SamplerV2 returns finite-shot bitstring counts rather than an analytic state. Independent random generators need not produce identical dictionaries.

In [2]:
circuit = QuantumCircuit(3)
circuit.h(0)
circuit.cx(0, 1)
circuit.cx(1, 2)
circuit.measure_all()
shots = 4096

def run_reference():
    result = StatevectorSampler(seed=19).run([circuit], shots=shots).result()[0]
    return result.data.meas.get_counts()

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(run_reference)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
backend = MettleQBackend(method="statevector", device="cpu")
compiled = transpile(circuit, backend, optimization_level=1)

def run_mettleq():
    result = MettleQSamplerV2(backend=backend).run([compiled], shots=shots).result()[0]
    return result.data.meas.get_counts()

candidate, mettleq_ms, _ = benchmark(run_mettleq)
tvd = total_variation_distance(reference, candidate)
support_ok = set(reference) <= {"000", "111"} and set(candidate) <= {"000", "111"}
method, device = qiskit_selection(backend)

## 4. Check correctness before discussing speed

Both distributions must have GHZ support and their total-variation distance must stay below the declared threshold.

In [5]:
tutorial_result = emit_result(
    notebook="qiskit/03_sampler_and_counts.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="finite-shot total-variation distance <= 0.05",
    passed=support_ok and tvd <= 0.05,
    exact_match=reference == candidate,
    selected_method=method,
    selected_device=device,
    metrics={"tvd": tvd, "reference_counts": reference, "mettleq_counts": candidate},
    notes="Independent sampler RNGs are compared statistically, not byte-for-byte.",
)


Comparison summary
------------------
Correctness contract: PASS — finite-shot total-variation distance <= 0.05
SDK reference median: 6.213 ms
MettleQ median:       8.729 ms
Timing interpretation: the SDK reference was 1.405x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)
Note: Independent sampler RNGs are compared statistically, not byte-for-byte.

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "finite-shot total-variation distance <= 0.05", "exact_match": false, "framework": "qiskit", "machine": "arm64", "metrics": {"mettleq_counts": {"000": 2044, "111": 2052}, "reference_counts": {"000": 2053, "111": 2043}, "tvd": 0.002197265625}, "mettleq_median_ms": 8.728667016839609, "notebook": "qiskit/03_sampler_and_counts.ipynb", "notes": "Independent sampler RNGs are compared statistically, not byte-for-byte.", "passed": true, "python": "3.13.2", "reference_median_ms": 6.

## What should you conclude?

Use sampling when the downstream program consumes counts. For very small circuits, SDK and RNG overhead dominate simulation time.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.